# Try ClawBio: your first pharmacogenomics demo

Run a real analysis on **bundled synthetic teaching data**, inspect its evidence, and see what happens when evidence is missing.

**You need:** a Google account and a standard CPU runtime. No AI account, API key, GPU or paid Colab plan is required for this exercise. Free Colab resources are subject to availability.

Select **Runtime > Run all**, or press each cell's play button from top to bottom. The first cell downloads code and dependencies; allow several minutes. Later analysis runs use bundled rules and make no evidence API requests. Colab may ask you to confirm running this public notebook.

**Learning goal:** a missing genotype is not evidence of a normal genotype.

This is a demonstration of pinned software behaviour, not a clinical validation. ClawBio is a research and educational tool. It is not a medical device and does not provide clinical diagnoses. Consult a healthcare professional before making any medical decisions.

[Full lesson](https://docs.clawbio.ai/tutorials/run-your-first-skill/)

## 1. Prepare the sandbox

Run this once. It downloads a fixed ClawBio revision and installs its locked dependencies into a separate environment. No access to Google Drive or personal genetic data is needed. A failed setup stops with an error instead of claiming success.

In [ ]:
#@title Prepare ClawBio
import hashlib
import json
import re
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from IPython.display import Markdown, FileLink, display

CLAWBIO_COMMIT = "7290841dfc9c7e817c12af38a2dd1479f0babe8e"
UV_VERSION = "0.10.1"
repo = Path(tempfile.gettempdir()) / ("clawbio-tutorial-" + CLAWBIO_COMMIT)

def checked(command, **kwargs):
    try:
        return subprocess.run(command, check=True, text=True, **kwargs)
    except subprocess.CalledProcessError as error:
        print(error.stderr or error.stdout or str(error))
        raise

print("Preparing the pinned environment. The first download may take several minutes.")
if not repo.exists():
    repo.mkdir()
    checked(["git", "init", "--quiet", str(repo)])
    checked(["git", "-C", str(repo), "remote", "add", "origin", "https://github.com/ClawBio/ClawBio.git"])
# Retry a previously interrupted download safely.
head = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True)
if head.returncode != 0:
    checked(["git", "-C", str(repo), "fetch", "--quiet", "--depth", "1", "origin", CLAWBIO_COMMIT])
    checked(["git", "-C", str(repo), "checkout", "--quiet", "--detach", CLAWBIO_COMMIT])
assert checked(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True).stdout.strip() == CLAWBIO_COMMIT
assert not checked(["git", "-C", str(repo), "diff", "HEAD", "--"], capture_output=True).stdout, "The tutorial checkout was edited. Restart with a fresh runtime."
checked([sys.executable, "-m", "pip", "install", "--quiet", "uv==" + UV_VERSION])
uv = [sys.executable, "-m", "uv"]
checked(uv + ["sync", "--locked", "--no-dev", "--python", "3.12", "--project", str(repo)], capture_output=True)
skill = repo / "skills" / "pharmgx-reporter" / "pharmgx_reporter.py"
demo_input = skill.parent / "demo_patient.txt"
assert skill.is_file() and demo_input.is_file()
run_root = Path(tempfile.mkdtemp(prefix="clawbio-demo-", dir=Path.cwd()))
print("Ready. ClawBio revision:", CLAWBIO_COMMIT)
print("Your results will be saved in:", run_root)

## 2. Run the demo

This executes PharmGx using the notebook's prepared environment. Online ClinPGx enrichment is explicitly disabled. Each run gets its own output folder, so rerunning a cell preserves earlier results.

In [ ]:
#@title Run the bundled PharmGx demo
run_commands = []

def run_demo(input_path, label):
    output = Path(tempfile.mkdtemp(prefix=label + "-", dir=run_root))
    command = uv + ["run", "--locked", "--no-dev", "--project", str(repo), "python", str(skill),
                    "--input", str(input_path), "--output", str(output), "--no-enrich"]
    completed = checked(command, capture_output=True)
    run_commands.append(command)
    # Present reports without decorative separators; leave numerical content intact.
    for name in ("report.md", "report.html"):
        path = output / name
        content = path.read_text()
        content = re.sub(r"(?m)^[ \t]*(?:---+|\*\*\*+|___+)[ \t]*$", "", content)
        content = re.sub(r"<hr\b[^>]*>", "", content, flags=re.I)
        content = content.replace("\u2014", "-").replace("\u2013", "-")
        if name == "report.md" and "not a medical device" not in content:
            content += "\n\nClawBio is a research and educational tool. It is not a medical device and does not provide clinical diagnoses. Consult a healthcare professional before making any medical decisions.\n"
        path.write_text(content)
    result = json.loads((output / "result.json").read_text())
    assert result["summary"]["clinpgx_enriched"] == 0
    print("Analysis complete:", output.name)
    return output, result

baseline_dir, baseline = run_demo(demo_input, "baseline")
display(Markdown((baseline_dir / "report.md").read_text()))

## 3. Inspect the evidence

These values come from this run's `result.json`, not a saved screenshot. The synthetic example has an ambiguous CYP2C19 allele combination. Check how the report expresses that uncertainty. The classifications demonstrate the pinned rules; they are not treatment advice.

In [ ]:
#@title Show computed results and check the demo
print(json.dumps(baseline["summary"], indent=2))
print("CYP2C19:", baseline["data"]["gene_profiles"]["CYP2C19"])
assert baseline["data"]["gene_profiles"]["CYP2C19"]["diplotype"].startswith("Indeterminate")
assert "not a medical device" in (baseline_dir / "report.md").read_text()
print("Checks passed: uncertainty is explicit and the research-use notice is present.")

## 4. What changes when evidence is missing?

Create a copy of the teaching input without the three CYP2C19 markers. The original file is preserved. Predict the result: should missing evidence produce a normal call, or an untested result?

This changes data availability, not a person's biology.

In [ ]:
#@title Remove CYP2C19 evidence and rerun
omitted_markers = {"rs4244285", "rs4986893", "rs12248560"}
changed_input = run_root / "teaching-input-without-cyp2c19.txt"
original_lines = demo_input.read_text().splitlines()
changed_input.write_text("\n".join(
    line for line in original_lines
    if not line.strip() or line.startswith("#") or line.split()[0] not in omitted_markers
) + "\n")
changed_dir, changed = run_demo(changed_input, "missing-evidence")
before = baseline["data"]["gene_profiles"]["CYP2C19"]
after = changed["data"]["gene_profiles"]["CYP2C19"]
print("Original teaching input:", before)
print("Without CYP2C19 markers:", after)
assert after["diplotype"] == "NOT_TESTED", "Missing markers must not become a normal genotype."
assert baseline["data"]["gene_profiles"]["DPYD"] == changed["data"]["gene_profiles"]["DPYD"]
print("Checks passed: missing CYP2C19 evidence is reported as NOT_TESTED; DPYD is unchanged.")

## 5. Download your evidence

The ZIP contains both reports, structured results, the modified teaching input and a tutorial provenance record with the pinned revision, exact commands and file checksums. Colab runtimes are temporary, so download results you want to keep.

The skill's own `reproducibility/commands.sh` is an upstream helper. For this offline tutorial, the exact commands including `--no-enrich` are in `tutorial-provenance.json`.

In [ ]:
#@title Package the results
provenance = {
    "commit": CLAWBIO_COMMIT,
    "uv_version": UV_VERSION,
    "analysis_mode": "bundled teaching data; online enrichment disabled",
    "commands": run_commands,
    "python": checked(uv + ["run", "--locked", "--no-dev", "--project", str(repo), "python", "--version"], capture_output=True).stdout.strip(),
    "original_input_sha256": hashlib.sha256(demo_input.read_bytes()).hexdigest(),
    "files_sha256": {
        str(path.relative_to(run_root)): hashlib.sha256(path.read_bytes()).hexdigest()
        for path in sorted(run_root.rglob("*")) if path.is_file() and path.name != "tutorial-provenance.json"
    },
}
(run_root / "tutorial-provenance.json").write_text(json.dumps(provenance, indent=2) + "\n")
bundle = Path(shutil.make_archive(str(run_root), "zip", root_dir=run_root))
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(bundle)))
else:
    files.download(str(bundle))
print("Evidence bundle ready:", bundle.name)

## You have completed the demo

You ran a skill, inspected its uncertainty, changed the available evidence, and verified that missing data was not treated as a normal genotype.

To repeat, rerun the cells. Each analysis creates a new output folder. If setup was interrupted, rerun setup first. For a fully clean start use **Runtime > Disconnect and delete runtime**, reconnect, then **Run all**.

[Return to the lesson](https://docs.clawbio.ai/tutorials/run-your-first-skill/) or [learn to build a skill](https://docs.clawbio.ai/tutorials/build-a-skill/).